In [1]:
import pandas as pd 
import numpy as np 
import plotly.express as px 
import plotly.graph_objects as go 
from scipy.stats import stats

In [2]:
pd.set_option("display.max_columns", None)

In [3]:
cols = ["Unit number (engine id)", "time_in_cycle", "operational setting 1", "operational setting 2", "operational setting 3"] + [   f"sensor_{i}"   for i in range(1,24)]

In [4]:
fd002_train = pd.read_csv(r"C:\potfolio\RUL_Prediction\data\raw\train_FD002.txt", header= None, sep = " ")
fd002_train.dropna(axis=1, how= "all")
fd002_train.columns = cols
fd002_train.head()

,Unit number (engine id),time_in_cycle,operational setting 1,operational setting 2,operational setting 3,sensor_1,sensor_2,sensor_3,sensor_4,sensor_5,sensor_6,sensor_7,sensor_8,sensor_9,sensor_10,sensor_11,sensor_12,sensor_13,sensor_14,sensor_15,sensor_16,sensor_17,sensor_18,sensor_19,sensor_20,sensor_21,sensor_22,sensor_23
0,1,1,34.9983,0.8400,100.0,449.44,555.32,1358.61,1137.23,5.48,8.00,194.64,2222.65,8341.91,1.02,42.02,183.06,2387.72,8048.56,9.3461,0.02,334,2223,100.00,14.73,8.8071,NaN,NaN
1,1,2,41.9982,0.8408,100.0,445.00,549.90,1353.22,1125.78,3.91,5.71,138.51,2211.57,8303.96,1.02,42.20,130.42,2387.66,8072.30,9.3774,0.02,330,2212,100.00,10.41,6.2665,NaN,NaN
2,1,3,24.9988,0.6218,60.0,462.54,537.31,1256.76,1047.45,7.05,9.02,175.71,1915.11,8001.42,0.94,36.69,164.22,2028.03,7864.87,10.8941,0.02,309,1915,84.93,14.08,8.6723,NaN,NaN
3,1,4,42.0077,0.8416,100.0,445.00,549.51,1354.03,1126.38,3.91,5.71,138.46,2211.58,8303.96,1.02,41.96,130.72,2387.61,8068.66,9.3528,0.02,329,2212,100.00,10.59,6.4701,NaN,NaN
4,1,5,25.0005,0.6203,60.0,462.54,537.07,1257.71,1047.93,7.05,9.03,175.05,1915.10,7993.23,0.94,36.89,164.31,2028.00,7861.23,10.8963,0.02,309,1915,84.93,14.13,8.5286,NaN,NaN


In [5]:
fd002_train[["sensor_22", "sensor_23"]].nunique()

sensor_22    0
sensor_23    0
dtype: int64

In [6]:
print(fd002_train["Unit number (engine id)"].nunique())
print(fd002_train["Unit number (engine id)"].min())
print(fd002_train["Unit number (engine id)"].max())

260
1
260


In [7]:
fd002_train = fd002_train.drop(columns=["sensor_22", "sensor_23"])

In [8]:
fd002_train.head()

,Unit number (engine id),time_in_cycle,operational setting 1,operational setting 2,operational setting 3,sensor_1,sensor_2,sensor_3,sensor_4,sensor_5,sensor_6,sensor_7,sensor_8,sensor_9,sensor_10,sensor_11,sensor_12,sensor_13,sensor_14,sensor_15,sensor_16,sensor_17,sensor_18,sensor_19,sensor_20,sensor_21
0,1,1,34.9983,0.8400,100.0,449.44,555.32,1358.61,1137.23,5.48,8.00,194.64,2222.65,8341.91,1.02,42.02,183.06,2387.72,8048.56,9.3461,0.02,334,2223,100.00,14.73,8.8071
1,1,2,41.9982,0.8408,100.0,445.00,549.90,1353.22,1125.78,3.91,5.71,138.51,2211.57,8303.96,1.02,42.20,130.42,2387.66,8072.30,9.3774,0.02,330,2212,100.00,10.41,6.2665
2,1,3,24.9988,0.6218,60.0,462.54,537.31,1256.76,1047.45,7.05,9.02,175.71,1915.11,8001.42,0.94,36.69,164.22,2028.03,7864.87,10.8941,0.02,309,1915,84.93,14.08,8.6723
3,1,4,42.0077,0.8416,100.0,445.00,549.51,1354.03,1126.38,3.91,5.71,138.46,2211.58,8303.96,1.02,41.96,130.72,2387.61,8068.66,9.3528,0.02,329,2212,100.00,10.59,6.4701
4,1,5,25.0005,0.6203,60.0,462.54,537.07,1257.71,1047.93,7.05,9.03,175.05,1915.10,7993.23,0.94,36.89,164.31,2028.00,7861.23,10.8963,0.02,309,1915,84.93,14.13,8.5286


In [13]:
fd002_train["max_life"] = fd002_train.groupby("Unit number (engine id)")["time_in_cycle"].transform("max")
fd002_train["RUL"]  = fd002_train["max_life"] - fd002_train["time_in_cycle"]
fd002_train.head()

,Unit number (engine id),time_in_cycle,operational setting 1,operational setting 2,operational setting 3,sensor_1,sensor_2,sensor_3,sensor_4,sensor_5,sensor_6,sensor_7,sensor_8,sensor_9,sensor_10,sensor_11,sensor_12,sensor_13,sensor_14,sensor_15,sensor_16,sensor_17,sensor_18,sensor_19,sensor_20,sensor_21,max_life,RUL
0,1,1,34.9983,0.8400,100.0,449.44,555.32,1358.61,1137.23,5.48,8.00,194.64,2222.65,8341.91,1.02,42.02,183.06,2387.72,8048.56,9.3461,0.02,334,2223,100.00,14.73,8.8071,149,148
1,1,2,41.9982,0.8408,100.0,445.00,549.90,1353.22,1125.78,3.91,5.71,138.51,2211.57,8303.96,1.02,42.20,130.42,2387.66,8072.30,9.3774,0.02,330,2212,100.00,10.41,6.2665,149,147
2,1,3,24.9988,0.6218,60.0,462.54,537.31,1256.76,1047.45,7.05,9.02,175.71,1915.11,8001.42,0.94,36.69,164.22,2028.03,7864.87,10.8941,0.02,309,1915,84.93,14.08,8.6723,149,146
3,1,4,42.0077,0.8416,100.0,445.00,549.51,1354.03,1126.38,3.91,5.71,138.46,2211.58,8303.96,1.02,41.96,130.72,2387.61,8068.66,9.3528,0.02,329,2212,100.00,10.59,6.4701,149,145
4,1,5,25.0005,0.6203,60.0,462.54,537.07,1257.71,1047.93,7.05,9.03,175.05,1915.10,7993.23,0.94,36.89,164.31,2028.00,7861.23,10.8963,0.02,309,1915,84.93,14.13,8.5286,149,144


In [15]:
fd002_train.drop(columns=["max_life"], inplace=True)

In [16]:
fd002_train[2300:2310]

,Unit number (engine id),time_in_cycle,operational setting 1,operational setting 2,operational setting 3,sensor_1,sensor_2,sensor_3,sensor_4,sensor_5,sensor_6,sensor_7,sensor_8,sensor_9,sensor_10,sensor_11,sensor_12,sensor_13,sensor_14,sensor_15,sensor_16,sensor_17,sensor_18,sensor_19,sensor_20,sensor_21,RUL
2300,12,134,0.0018,0.0000,100.0,518.67,642.30,1592.44,1405.78,14.62,21.61,553.94,2388.05,9053.98,1.30,47.27,521.73,2388.04,8138.05,8.4091,0.03,392,2388,100.00,39.01,23.3769,115
2301,12,135,24.9988,0.6200,60.0,462.54,537.05,1260.45,1052.69,7.05,9.03,175.73,1915.33,8017.24,0.94,36.64,164.73,2028.25,7870.80,10.8935,0.02,305,1915,84.93,14.10,8.5857,114
2302,12,136,0.0022,0.0000,100.0,518.67,641.52,1586.45,1406.14,14.62,21.61,554.03,2388.06,9055.89,1.30,47.54,521.77,2388.03,8138.41,8.4433,0.03,392,2388,100.00,38.93,23.3185,113
2303,12,137,25.0051,0.6205,60.0,462.54,536.78,1259.54,1051.73,7.05,9.03,175.38,1915.26,8010.99,0.94,36.68,164.84,2028.22,7868.04,10.8639,0.02,306,1915,84.93,14.28,8.6275,112
2304,12,138,25.0050,0.6219,60.0,462.54,536.06,1252.86,1057.22,7.05,9.03,175.60,1915.33,8020.53,0.94,36.71,164.77,2028.29,7874.09,10.8927,0.02,308,1915,84.93,14.31,8.6206,111
2305,12,139,20.0000,0.7000,100.0,491.19,607.37,1478.71,1259.07,9.35,13.66,335.13,2323.94,8728.23,1.07,44.36,314.82,2388.07,8062.67,9.2045,0.02,364,2324,100.00,24.52,14.6854,110
2306,12,140,35.0038,0.8400,100.0,449.44,555.75,1363.61,1128.50,5.48,8.00,194.25,2222.98,8356.23,1.02,41.90,183.00,2388.08,8069.26,9.3353,0.02,333,2223,100.00,14.79,8.9471,109
2307,12,141,25.0065,0.6217,60.0,462.54,536.73,1259.75,1046.94,7.05,9.02,175.22,1915.38,8011.52,0.94,36.75,164.57,2028.27,7874.82,10.8837,0.02,309,1915,84.93,14.46,8.6003,108
2308,12,142,42.0036,0.8411,100.0,445.00,549.20,1354.39,1121.61,3.91,5.71,138.88,2211.94,8321.58,1.02,41.94,130.82,2387.96,8081.89,9.3492,0.02,330,2212,100.00,10.57,6.3758,107
2309,12,143,10.0029,0.2500,100.0,489.05,605.05,1503.82,1308.79,10.52,15.49,395.05,2318.87,8779.05,1.26,45.32,371.91,2388.10,8126.08,8.6155,0.03,368,2319,100.00,28.66,17.1078,106


In [18]:
fig = px.histogram(fd002_train, x="time_in_cycle", nbins= 25, title = "remaining life distribution")
fig.update_traces(marker = dict(color = "yellow", line = dict(color = "black", width = 1)))
fig.show()

In [20]:
print("Number of unique engine : ")
print(fd002_train["Unit number (engine id)"].nunique())
print("Max remaning life : ")
print(fd002_train["time_in_cycle"].max())
print(fd002_train["RUL"].describe())

Number of unique engine : 
260
Max remaning life : 
378
count    53759.000000
mean       108.154746
std         69.180569
min          0.000000
25%         51.000000
50%        103.000000
75%        156.000000
max        377.000000
Name: RUL, dtype: float64


In [22]:
max_cycle_per_engine = fd002_train.groupby("Unit number (engine id)")["time_in_cycle"].max()

fig = px.histogram(max_cycle_per_engine, x = max_cycle_per_engine, nbins= 25, title= "Distirbution of remaing life across engines")
fig.update_traces(marker = dict(color = "skyblue", line = dict(color = "black", width = 1)))
fig.show()

In [24]:
sensor_col = [  col  for col in fd002_train.columns if "sensor" in col ]
print(sensor_col)

['sensor_1', 'sensor_2', 'sensor_3', 'sensor_4', 'sensor_5', 'sensor_6', 'sensor_7', 'sensor_8', 'sensor_9', 'sensor_10', 'sensor_11', 'sensor_12', 'sensor_13', 'sensor_14', 'sensor_15', 'sensor_16', 'sensor_17', 'sensor_18', 'sensor_19', 'sensor_20', 'sensor_21']


In [27]:
correlation_rul = fd002_train[sensor_col + ["RUL"]].corr()["RUL"].sort_values(ascending=True)
correlation_rul

sensor_16   -0.071352
sensor_11   -0.046950
sensor_14   -0.042325
sensor_4    -0.040978
sensor_15   -0.038455
sensor_17   -0.027031
sensor_3    -0.026942
sensor_9    -0.015306
sensor_2    -0.004928
sensor_5    -0.000758
sensor_6    -0.000496
sensor_1    -0.000023
sensor_12    0.002249
sensor_7     0.002430
sensor_10    0.004306
sensor_8     0.004345
sensor_18    0.004780
sensor_13    0.005245
sensor_19    0.005761
sensor_21    0.006165
sensor_20    0.006287
RUL          1.000000
Name: RUL, dtype: float64